# Getting kretsinger off an air-gapped machine

The premise: a machine holding this repo has no network, no USB, no git. The only
way out is its screen. This notebook works out exactly what to send, what to leave
behind, and how many photographs it costs.

Run it from `kretsinger/airgap-qr-transfer/`; `ROOT` resolves to the repo above it.

Needs `pip install segno`. Decoding on the receiving machine additionally needs
`pip install pyzbar pillow` plus `brew install zbar`.

> The saved outputs below were produced against a copy of this repo taken while the
> notebook was written. Re-run and the numbers move with whatever the repo holds now.

In [1]:
import pathlib, agmini

ROOT = pathlib.Path("..").resolve()
print("repo:", ROOT)
print("agmini:", agmini.__file__)

repo: /home/claude/kret
agmini: /home/claude/kret/airgap-qr-transfer/agmini.py


## 1. What happens with no filters at all

Start with the naive call. `collect()` reads nothing into memory beyond a size check —
it just tells you what `pack()` would send.

In [ ]:
everything = agmini.collect(ROOT)
total = sum((ROOT / f).stat().st_size for f in everything)
print(f"{len(everything)} files, {total/1024:,.0f} KB")

from collections import Counter

ext = Counter(pathlib.Path(f).suffix or "(none)" for f in everything)
for e, n in ext.most_common(10):
    size = sum((ROOT / f).stat().st_size for f in everything if (pathlib.Path(f).suffix or "(none)") == e)
    print(f"  {e:<10} {n:>4} files  {size/1024:>9,.0f} KB")

263 files, 3,441 KB
  .py         173 files        427 KB
  .ipynb       41 files      2,953 KB
  .bash        15 files         19 KB
  .md          13 files          8 KB
  .zsh         13 files         28 KB
  .sh           3 files          4 KB
  .toml         2 files          1 KB
  .txt          2 files          0 KB
  (none)        1 files          1 KB


## 2. Look at what that would put on a screen

This is the step worth doing slowly. Everything in the list gets rendered as QR codes
and photographed, so anything sensitive in there ends up in your camera roll.

In [ ]:
import fnmatch

SUSPECT = [
    "*token*",
    "*secret*",
    "*credential*",
    "*api*",
    "*key*",
    "*.env*",
    "vault/*",
    "*password*",
    "*.pem",
    "*.cfg",
    "*rc",
]
flagged = [
    f
    for f in everything
    if any(fnmatch.fnmatch(f.lower(), p) or fnmatch.fnmatch(pathlib.Path(f).name.lower(), p) for p in SUSPECT)
]
print("worth a second look:")
for f in flagged:
    print(f"   {f}")
if not flagged:
    print("   (none in this staged copy)")

worth a second look:
   bash_scripts/.bashrc


**The one that matters: `vault/`.**

The real repo has a `vault/` directory holding `hugging-face-token.txt`,
`fred-api.txt`, `eia-api.txt`, `claude-code.txt` and `source_tokens.zsh`. agmini's
built-in secret patterns **do not catch these** — the pattern is `*_token`, and
`hugging-face-token.txt` ends in `.txt`, so it slips through. `.env` is caught;
`vault/` is not. Run section 2 against the real repo and those five files appear in
the list — they are absent from the saved output below only because they were left
out of the copy this notebook was executed against.

So exclude it explicitly. Two options, and the second is the durable one:

```python
# per-call
agmini.collect(ROOT, exclude=["vault/*"])

# permanent - edit the module so you can never forget
agmini.SECRETS = agmini.SECRETS + ("vault/*", "*token*", "*-api.txt")
```

The `bash_scripts/.bashrc` and `bashrc.d/*` files are worth a glance too — shell
config often carries exported keys. Skim them before packing rather than after
photographing.

In [ ]:
# make the exclusion permanent for this session
agmini.SECRETS = agmini.SECRETS + ("vault/*", "*token*", "*-api.txt")

safe = agmini.collect(ROOT)
print("removed by the tightened patterns:", sorted(set(everything) - set(safe)) or "(nothing in this staged copy)")
print(f"\n{len(safe)} files remain")

removed by the tightened patterns: (nothing in this staged copy)

263 files remain


## 3. Code only, versus code plus notebooks

The `.ipynb` files are where the cost lives, and it is not close.

In [ ]:
import io, lzma, tarfile


def payload_size(files):
    """Exactly what pack() would compress, without writing anything."""
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w") as tar:
        for rel in files:
            blob = (ROOT / rel).read_bytes()
            info = tarfile.TarInfo(rel)
            info.size, info.mtime = len(blob), 0
            tar.addfile(info, io.BytesIO(blob))
    return len(lzma.compress(buf.getvalue(), preset=9))


CODE = [".py", ".sh", ".zsh", ".bash", ".toml", ".yaml", ".yml", ".md", ".txt"]
code_only = agmini.collect(ROOT, include=CODE)
with_nbs = agmini.collect(ROOT, include=CODE + [".ipynb"])

# ~4.6 KB per sheet at the 1080p defaults; ~8.3 KB at 1440p with chunk=1100
print(f"{'selection':<22}{'files':>7}{'raw KB':>10}{'xz KB':>9}{'photos 1080p':>14}{'1440p':>8}")
print("-" * 70)
for label, files in [("code only", code_only), ("code + notebooks", with_nbs)]:
    raw = sum((ROOT / f).stat().st_size for f in files)
    xz = payload_size(files)
    print(f"{label:<22}{len(files):>7}{raw/1024:>10,.0f}{xz/1024:>9,.0f}" f"{-(-xz//4710):>14}{-(-xz//8500):>8}")

selection               files    raw KB    xz KB  photos 1080p   1440p
----------------------------------------------------------------------


code only                 221       487      106            24      13


code + notebooks          262     3,441      810           177      98


## 4. Clearing notebook outputs

Most of that `.ipynb` weight is stored cell output — rendered plots, base64 PNGs,
dumped dataframes — none of which is source. Clearing it costs you nothing you can't
regenerate by re-running.

In [ ]:
import json


def cleared_bytes(rel):
    nb = json.loads((ROOT / rel).read_text())
    for c in nb.get("cells", []):
        c["outputs"] = []
        c["execution_count"] = None
    return json.dumps(nb, separators=(",", ":")).encode()


nbs = [f for f in safe if f.endswith(".ipynb")]
raw_nb = sum((ROOT / f).stat().st_size for f in nbs)
clean_nb = sum(len(cleared_bytes(f)) for f in nbs)

# compress the cleared versions together, the way pack() would
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w") as tar:
    for rel in nbs:
        blob = cleared_bytes(rel)
        info = tarfile.TarInfo(rel)
        info.size, info.mtime = len(blob), 0
        tar.addfile(info, io.BytesIO(blob))
xz_clean = len(lzma.compress(buf.getvalue(), preset=9))
xz_raw = payload_size(nbs)

print(f"{len(nbs)} notebooks")
print(
    f"  as-is            {raw_nb/1024:>8,.0f} KB raw -> {xz_raw/1024:>7,.0f} KB xz" f" -> {-(-xz_raw//4710):>4} photos"
)
print(
    f"  outputs cleared  {clean_nb/1024:>8,.0f} KB raw -> {xz_clean/1024:>7,.0f} KB xz"
    f" -> {-(-xz_clean//4710):>4} photos"
)
print(f"\n  {xz_raw/max(xz_clean,1):.0f}x less to photograph")

41 notebooks
  as-is               2,953 KB raw ->     708 KB xz ->  154 photos
  outputs cleared       267 KB raw ->      34 KB xz ->    8 photos

  21x less to photograph


On the offline machine, before packing:

```bash
jupyter nbconvert --clear-output --inplace **/*.ipynb
```

No jupyter there? Standard library only, and **copy the tree first** — this rewrites
in place:

```python
import json, glob
for f in glob.glob("**/*.ipynb", recursive=True):
    nb = json.load(open(f))
    for c in nb.get("cells", []):
        c["outputs"] = []; c["execution_count"] = None
    json.dump(nb, open(f, "w"))
```

## 5. What to leave behind entirely

`kretsinger/data/` holds a 253 MB `.parquet`. At roughly 4.6 KB per photo that is
over a million photographs — treat data as non-transferable and regenerate it on the
far side. agmini skips it anyway: the 1 MiB `max_size` cut and the binary check both
exclude it, which is why it never appeared in any list above.

## 6. Pack it

In [ ]:
info = agmini.pack(
    ROOT,
    out=ROOT.parent / "kretsinger-sheets.html",  # outside the repo
    include=CODE,
    exclude=["vault/*", "scratch/*", "example_notebooks_internet/*"],
)
for k, v in info.items():
    print(f"  {k:<18} {v}")

  path               /home/claude/kretsinger-sheets.html
  files              220
  raw_bytes          498627
  compressed_bytes   116676
  frames             196
  sheets             25
  sha256             8df717679da9353524621b6c457dddda7e689a556510f6d6008c53eb1290cd11
  transfer_id        8df71767


`sheets` is the number of photographs. Open the HTML fullscreen and shoot one screen
at a time — order doesn't matter, duplicates don't matter, every frame carries its own
index and checksum.

On a 1440p display, ask for bigger symbols and it drops further:

In [ ]:
for label, kw in [
    ("1080p defaults ", {}),
    ("1440p          ", dict(chunk=1100)),
    ("4K             ", dict(chunk=1100, cols=6, rows=3)),
]:
    r = agmini.pack(ROOT, out="/tmp/_t.html", include=CODE, exclude=["vault/*", "scratch/*"], **kw)
    print(f"  {label} {r['sheets']:>3} photos   ({r['frames']} frames)")

  1080p defaults   25 photos   (196 frames)


  1440p            14 photos   (108 frames)


  4K                6 photos   (108 frames)


## 7. Check it before you photograph anything

The frames are text, so the encoding can be verified without a camera. Worth running:
a mistake found here costs seconds, the same mistake found after capture costs the
whole session.

In [ ]:
import base64, hashlib, struct, zlib, re

html = pathlib.Path(info["path"]).read_text()
print("symbols drawn:", html.count("<div class=c>"), "of", info["frames"], "frames")

# pull the base32 payloads back out of the frame texts agmini built
sel = agmini.collect(ROOT, include=CODE, exclude=["vault/*", "scratch/*", "example_notebooks_internet/*"])
print("files packed :", len(sel))
print("sha256       :", info["sha256"][:48], "...")

again = agmini.pack(
    ROOT, out="/tmp/_again.html", include=CODE, exclude=["vault/*", "scratch/*", "example_notebooks_internet/*"]
)
print("deterministic:", again["sha256"] == info["sha256"])

symbols drawn: 196 of 196 frames
files packed : 220
sha256       : 8df717679da9353524621b6c457dddda7e689a556510f6d6 ...


deterministic: True


## 8. The actual procedure

**On the offline machine**

1. Get a sender there. `agtype.py` is 87 lines and exists to be typed by hand, since
   that machine cannot receive files. `agmini.py` is better if you can get a file
   across somehow.
2. `pip list | grep segno` — without it, nothing here runs.
3. Clear notebook outputs (section 4), if you're sending any.
4. Plug in the biggest monitor that machine can drive. Throughput scales with the
   sending display, not the phone: 1080p → 1440p roughly halves the photo count.
5. `agmini.pack(root, "sheets.html", include=[...], exclude=["vault/*"])`
6. Open it fullscreen. One photo per screen, square-on, tap to focus, no glare.
   Check the first photo is sharp before shooting forty more.

**Back here**

```bash
python3 agunpack.py decode ~/photos/kretsinger --out ~/kretsinger-recovered
```

It reports missing frames as *sheets to re-shoot*, caches what it already has, and
verifies a SHA-256 over the whole payload plus one per file before it writes anything.
Then `cd ~/kretsinger-recovered && git init`.

`demo.ipynb` walks the capture-and-decode half in detail, with simulated photographs.